# Day 3 — Stage 4: Layout Analysis

**Goal:** take the flat list of OCR results (words or lines with bounding boxes) 
and group them into logical structure: lines, then regions (header, items, totals, footer).

**Why this is a separate stage:** OCR engines give us text + positions, but no understanding 
of what goes together. "TOTAL" and "38.37" are separate detections that happen to be on the 
same line. Layout analysis connects them — "TOTAL is a label, 38.37 is its value, they belong 
together because they share the same vertical position."

**Approach:** pure geometry over bounding boxes. No ML, no libraries — just spatial rules 
we can explain and justify.

In [1]:
import cv2
import sys
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.preprocessing import preprocess
from src.ocr import run_tesseract, run_paddleocr

# Load image and run both engines
image_path = project_root / "sample_images" / "sroie_05.jpg"
image = cv2.imread(str(image_path))
preprocessed = preprocess(image)

tess_results = run_tesseract(preprocessed)
paddle_results = run_paddleocr(image_path)

print(f"Tesseract: {len(tess_results)} words")
print(f"PaddleOCR: {len(paddle_results)} lines")
print(f"\nSample Tesseract result: {tess_results[3]}")
print(f"Sample PaddleOCR result: {paddle_results[1]}")

Tesseract: 150 words
PaddleOCR: 54 lines

Sample Tesseract result: {'text': 'Guardian', 'conf': 0.92, 'bbox': (33, 285, 96, 26)}
Sample PaddleOCR result: {'text': 'Guardian Health And Heauty Sdn Bhd', 'conf': 0.928707480430603, 'bbox': (32, 283, 412, 29)}


In [2]:
def group_into_lines(ocr_results, y_threshold=10):
    """
    Group OCR results into lines based on vertical position.
    
    Words whose vertical centers are within y_threshold pixels of each 
    other are considered to be on the same line.
    
    Args:
        ocr_results: list of dicts with 'text', 'conf', 'bbox' (x, y, w, h)
        y_threshold: max vertical distance (pixels) between words on same line
    
    Returns:
        List of lines, each line is a list of word-dicts sorted left-to-right.
        Lines are sorted top-to-bottom.
    """
    if not ocr_results:
        return []
    
    # Calculate vertical center for each result
    items = []
    for r in ocr_results:
        x, y, w, h = r["bbox"]
        y_center = y + h / 2
        items.append({"result": r, "y_center": y_center})
    
    # Sort by vertical position (top to bottom)
    items.sort(key=lambda item: item["y_center"])
    
    # Group into lines: walk through sorted items, start a new line 
    # whenever the y_center jumps by more than y_threshold
    lines = []
    current_line = [items[0]]
    
    for item in items[1:]:
        if abs(item["y_center"] - current_line[-1]["y_center"]) <= y_threshold:
            current_line.append(item)
        else:
            lines.append(current_line)
            current_line = [item]
    lines.append(current_line)  # don't forget the last line
    
    # Sort words within each line left-to-right by x coordinate
    # and unwrap back to just the result dicts
    sorted_lines = []
    for line in lines:
        line.sort(key=lambda item: item["result"]["bbox"][0])
        sorted_lines.append([item["result"] for item in line])
    
    return sorted_lines


# Test on Tesseract results
tess_lines = group_into_lines(tess_results)

print(f"150 words grouped into {len(tess_lines)} lines\n")
print("First 10 lines:")
for i, line in enumerate(tess_lines[:10]):
    text = " ".join([w["text"] for w in line])
    y_pos = line[0]["bbox"][1]
    print(f"  Line {i+1:>2} (y={y_pos:>4}): {text}")

150 words grouped into 37 lines

First 10 lines:
  Line  1 (y= 181): Treple ted LOc8,
  Line  2 (y= 285): Guardian Health And Beauty Sdn Phd
  Line  3 (y= 316): Jalan Loke Yew Bentong
  Line  4 (y= 352): 90 Ground Floor
  Line  5 (y= 382): Jalan Bentong
  Line  6 (y= 417): Tel:09-222 6498
  Line  7 (y= 449): Company Reg #1101083-T
  Line  8 (y= 484): GST Reo HOOORS9B748L4
  Line  9 (y= 564): .
  Line 10 (y= 583): 121093507 G BX TISSU 4X15) * 6.88 §


In [3]:
def identify_regions(lines, image_height):
    """
    Split grouped lines into receipt regions based on vertical position.
    
    Rough splits:
        Header: top 25% of image
        Items:  25% to 55%
        Totals: 55% to 75%
        Footer: bottom 25%
    
    These are starting estimates — we'll refine based on content cues later.
    
    Args:
        lines: output of group_into_lines (list of line-lists)
        image_height: height of the image in pixels
    
    Returns:
        Dict with keys 'header', 'items', 'totals', 'footer',
        each containing a list of lines.
    """
    regions = {"header": [], "items": [], "totals": [], "footer": []}
    
    for line in lines:
        # Use the first word's y position as the line's vertical position
        y = line[0]["bbox"][1]
        relative_y = y / image_height
        
        if relative_y < 0.25:
            regions["header"].append(line)
        elif relative_y < 0.55:
            regions["items"].append(line)
        elif relative_y < 0.75:
            regions["totals"].append(line)
        else:
            regions["footer"].append(line)
    
    return regions


# Test
image_height = image.shape[0]
regions = identify_regions(tess_lines, image_height)

print(f"Image height: {image_height}px\n")
for region_name, region_lines in regions.items():
    print(f"--- {region_name.upper()} ({len(region_lines)} lines) ---")
    for line in region_lines:
        text = " ".join([w["text"] for w in line])
        print(f"  {text}")
    print()

Image height: 1646px

--- HEADER (5 lines) ---
  Treple ted LOc8,
  Guardian Health And Beauty Sdn Phd
  Jalan Loke Yew Bentong
  90 Ground Floor
  Jalan Bentong

--- ITEMS (13 lines) ---
  Tel:09-222 6498
  Company Reg #1101083-T
  GST Reo HOOORS9B748L4
  .
  121093507 G BX TISSU 4X15) * 6.88 §
  2 6.88
  121087265 HK BX TISS 04 = * «13,76 8
  121096625 GON CHINT H/WASHPS = 12.90 §
  121096057 GDN KTCH LILYSOOKL = 6.90 8
  PUP 121096057 2.07-
  SUBTOTAL 38.37
  TOTAL <G8T INCL) 38.37
  CASH 40.35

--- TOTALS (8 lines) ---
  ROUNDING ADJUSTMENTS 0.02-
  CHANGE DUE 2,00
  YOUR SAVINGS FOR TODAY 1.5
  6ST - Rate -—— 6ST Excl ~- GST Ant
  § &h %.20 00 2.17
  ~ Prono price items
  |
  Thank You For Shopping

--- FOOTER (11 lines) ---
  At Guardian
  All Attounts Are in RH
  Goods sold are non-refundable.
  Dispensed medicines sold are
  not returnable.
  ALL exchanges and returns including
  Guardian brand products must be nade
  within ?days with original receipts and
  product in origina

In [4]:
def identify_regions(lines, image_height):
    """
    Split lines into receipt regions using keyword anchors + position.
    
    Strategy:
        1. Find the SUBTOTAL/TOTAL line — that's the boundary between items and totals
        2. Everything above it: split into header vs items using vendor-info keywords
        3. Everything at or below it: split into totals vs footer using 
           thank-you / policy keywords
    
    Falls back to position-based splits if no keywords found.
    """
    # Keywords that signal each region
    header_keywords = {"tel", "phone", "fax", "reg", "gst", "company", "address"}
    totals_keywords = {"subtotal", "total", "cash", "change", "rounding", "visa",
                       "mastercard", "payment", "tendered", "balance"}
    footer_keywords = {"thank", "exchange", "return", "refund", "receipt",
                       "condition", "warranty", "goods", "medicine"}
    
    def line_text(line):
        return " ".join([w["text"] for w in line]).lower()
    
    def line_has_keyword(line, keywords):
        text = line_text(line)
        return any(kw in text for kw in keywords)
    
    # Step 1: find the first line containing a totals keyword
    totals_start = None
    for i, line in enumerate(lines):
        if line_has_keyword(line, {"subtotal", "total"}):
            totals_start = i
            break
    
    # Step 2: find where footer starts (first thank/exchange/return AFTER totals)
    footer_start = None
    search_from = totals_start + 1 if totals_start is not None else len(lines) // 2
    for i in range(search_from, len(lines)):
        if line_has_keyword(lines[i], footer_keywords):
            footer_start = i
            break
    
    # Step 3: find where header ends (last header-keyword line before items)
    header_end = 0
    for i, line in enumerate(lines):
        if totals_start is not None and i >= totals_start:
            break
        if line_has_keyword(line, header_keywords):
            header_end = i + 1  # header includes this line
    
    # If no header keywords found, use top 20% as fallback
    if header_end == 0:
        for i, line in enumerate(lines):
            y = line[0]["bbox"][1]
            if y / image_height > 0.20:
                header_end = i
                break
    
    # If no totals keyword found, use 60% as fallback
    if totals_start is None:
        for i, line in enumerate(lines):
            y = line[0]["bbox"][1]
            if y / image_height > 0.60:
                totals_start = i
                break
        if totals_start is None:
            totals_start = len(lines)
    
    # If no footer keyword found, use bottom 25% as fallback
    if footer_start is None:
        for i, line in enumerate(lines):
            y = line[0]["bbox"][1]
            if y / image_height > 0.75:
                footer_start = i
                break
        if footer_start is None:
            footer_start = len(lines)
    
    regions = {
        "header": lines[:header_end],
        "items":  lines[header_end:totals_start],
        "totals": lines[totals_start:footer_start],
        "footer": lines[footer_start:],
    }
    
    return regions


# Test
regions = identify_regions(tess_lines, image_height)

for region_name, region_lines in regions.items():
    print(f"--- {region_name.upper()} ({len(region_lines)} lines) ---")
    for line in region_lines:
        text = " ".join([w["text"] for w in line])
        print(f"  {text}")
    print()

--- HEADER (8 lines) ---
  Treple ted LOc8,
  Guardian Health And Beauty Sdn Phd
  Jalan Loke Yew Bentong
  90 Ground Floor
  Jalan Bentong
  Tel:09-222 6498
  Company Reg #1101083-T
  GST Reo HOOORS9B748L4

--- ITEMS (7 lines) ---
  .
  121093507 G BX TISSU 4X15) * 6.88 §
  2 6.88
  121087265 HK BX TISS 04 = * «13,76 8
  121096625 GON CHINT H/WASHPS = 12.90 §
  121096057 GDN KTCH LILYSOOKL = 6.90 8
  PUP 121096057 2.07-

--- TOTALS (10 lines) ---
  SUBTOTAL 38.37
  TOTAL <G8T INCL) 38.37
  CASH 40.35
  ROUNDING ADJUSTMENTS 0.02-
  CHANGE DUE 2,00
  YOUR SAVINGS FOR TODAY 1.5
  6ST - Rate -—— 6ST Excl ~- GST Ant
  § &h %.20 00 2.17
  ~ Prono price items
  |

--- FOOTER (12 lines) ---
  Thank You For Shopping
  At Guardian
  All Attounts Are in RH
  Goods sold are non-refundable.
  Dispensed medicines sold are
  not returnable.
  ALL exchanges and returns including
  Guardian brand products must be nade
  within ?days with original receipts and
  product in original condition.
  St:8303

In [5]:
def pair_label_value(line):
    """
    Split a line into label (left side) and value (right side).
    
    Strategy: the rightmost word(s) that look numeric are the value.
    Everything to the left is the label.
    
    Args:
        line: list of word-dicts from one line
    
    Returns:
        (label_text, value_text) tuple. value_text may be empty if no 
        numeric content found on the right side.
    """
    if not line:
        return ("", "")
    
    # Walk from right to left, collecting words that look numeric
    value_words = []
    label_words = []
    found_value = False
    
    for word in reversed(line):
        text = word["text"]
        
        # Does this word look like a number or price?
        # Strip common symbols first
        cleaned = text.replace(",", "").replace(".", "").replace("-", "")
        cleaned = cleaned.replace("$", "").replace("§", "").replace("*", "")
        cleaned = cleaned.replace("=", "").replace("~", "").replace("|", "")
        
        if not found_value and (cleaned.isdigit() or text in [".", "-"]):
            value_words.insert(0, text)
            found_value = True
        elif found_value and (cleaned.isdigit() or text in [".", "-", "*", "="]):
            value_words.insert(0, text)
        else:
            label_words.insert(0, text)
    
    label = " ".join(label_words).strip()
    value = " ".join(value_words).strip()
    
    # Clean up common OCR artifacts in the value
    value = value.replace("§", "").replace("*", "").replace("=", "")
    value = value.replace("|", "").replace("~", "").strip()
    
    return label, value


# Test on totals region
print("Label-Value pairs from TOTALS region:\n")
for line in regions["totals"]:
    full_text = " ".join([w["text"] for w in line])
    label, value = pair_label_value(line)
    print(f"  Full line:  {full_text}")
    print(f"  Label:      '{label}'")
    print(f"  Value:      '{value}'")
    print()

Label-Value pairs from TOTALS region:

  Full line:  SUBTOTAL 38.37
  Label:      'SUBTOTAL'
  Value:      '38.37'

  Full line:  TOTAL <G8T INCL) 38.37
  Label:      'TOTAL <G8T INCL)'
  Value:      '38.37'

  Full line:  CASH 40.35
  Label:      'CASH'
  Value:      '40.35'

  Full line:  ROUNDING ADJUSTMENTS 0.02-
  Label:      'ROUNDING ADJUSTMENTS'
  Value:      '0.02-'

  Full line:  CHANGE DUE 2,00
  Label:      'CHANGE DUE'
  Value:      '2,00'

  Full line:  YOUR SAVINGS FOR TODAY 1.5
  Label:      'YOUR SAVINGS FOR TODAY'
  Value:      '1.5'

  Full line:  6ST - Rate -—— 6ST Excl ~- GST Ant
  Label:      '6ST Rate -—— 6ST Excl ~- GST Ant'
  Value:      '-'

  Full line:  § &h %.20 00 2.17
  Label:      '§ &h %.20'
  Value:      '00 2.17'

  Full line:  ~ Prono price items
  Label:      '~ Prono price items'
  Value:      ''

  Full line:  |
  Label:      '|'
  Value:      ''



In [6]:
import importlib
import src.layout
importlib.reload(src.layout)

from src.layout import group_into_lines as gli, identify_regions as ir, pair_label_value as plv

test_lines = gli(tess_results)
test_regions = ir(test_lines, image_height)

print(f"Lines: {len(test_lines)} (expected {len(tess_lines)})")
print(f"Header: {len(test_regions['header'])} lines")
print(f"Items:  {len(test_regions['items'])} lines")
print(f"Totals: {len(test_regions['totals'])} lines")
print(f"Footer: {len(test_regions['footer'])} lines")

# Quick label-value check
label, value = plv(test_regions["totals"][0])
print(f"\nFirst totals line: '{label}' → '{value}'")

Lines: 37 (expected 37)
Header: 8 lines
Items:  7 lines
Totals: 10 lines
Footer: 12 lines

First totals line: 'SUBTOTAL' → '38.37'


## Stage 5: Field Extraction

Pull structured fields from the layout regions using keyword matching 
and regex patterns. Each field has a list of synonyms (labels it might 
appear under) and a pattern for validating/cleaning the value.

**Required fields:** document_type, vendor_name, invoice_number/receipt_number, 
invoice_date, total_amount

**Good-to-have:** subtotal, tax_amount, currency, payment_mode

In [7]:
import re

def extract_fields(regions, ocr_results):
    """
    Extract structured fields from layout regions.
    
    Args:
        regions: dict from identify_regions (header/items/totals/footer)
        ocr_results: raw OCR results (for fallback full-text search)
    
    Returns:
        Dict with extracted fields and confidence notes.
    """
    fields = {}
    
    # --- VENDOR NAME ---
    # Usually the first non-garbage line in the header (skip handwriting)
    # Look for the first line with confidence > 0.5
    for line in regions["header"]:
        avg_conf = sum(w["conf"] for w in line) / len(line)
        text = " ".join([w["text"] for w in line])
        # Skip very short lines and low-confidence lines
        if avg_conf > 0.5 and len(text) > 5:
            fields["vendor_name"] = text
            break
    
    # --- TOTAL AMOUNT ---
    total_synonyms = {"total", "grand total", "amount payable", "net amount",
                      "total amount", "amount due", "total (gst incl)",
                      "total(gst incl)", "totalgst incl"}
    
    for line in regions["totals"]:
        line_text = " ".join([w["text"] for w in line]).lower()
        # Clean up OCR artifacts for matching
        line_clean = line_text.replace("<", "(").replace(")", ")").replace(">", ")")
        
        for synonym in total_synonyms:
            if synonym in line_clean:
                _, value = pair_label_value(line)
                if value:
                    fields["total_amount"] = value
                    break
        if "total_amount" in fields:
            break
    
    # --- SUBTOTAL ---
    subtotal_synonyms = {"subtotal", "sub total", "sub-total"}
    
    for line in regions["totals"]:
        line_text = " ".join([w["text"] for w in line]).lower()
        for synonym in subtotal_synonyms:
            if synonym in line_text:
                _, value = pair_label_value(line)
                if value:
                    fields["subtotal"] = value
                    break
        if "subtotal" in fields:
            break
    
    # --- DATE ---
    # Search ALL regions — date can appear in header or footer
    all_text = ""
    for line in regions["header"] + regions["footer"]:
        all_text += " ".join([w["text"] for w in line]) + " "
    
    # Common date patterns: DD/MM/YY, DD/MM/YYYY, DD-MM-YYYY, YYYY-MM-DD
    date_patterns = [
        r'\d{2}/\d{2}/\d{4}',    # DD/MM/YYYY
        r'\d{2}/\d{2}/\d{2}',    # DD/MM/YY
        r'\d{2}-\d{2}-\d{4}',    # DD-MM-YYYY
        r'\d{4}-\d{2}-\d{2}',    # YYYY-MM-DD
        r'\d{2}\.\d{2}\.\d{4}',  # DD.MM.YYYY
    ]
    
    for pattern in date_patterns:
        match = re.search(pattern, all_text)
        if match:
            fields["date"] = match.group()
            break
    
    # --- PAYMENT MODE ---
    payment_synonyms = {"cash", "visa", "mastercard", "credit card",
                        "debit card", "nets", "amex", "upi", "card"}
    
    for line in regions["totals"]:
        line_text = " ".join([w["text"] for w in line]).lower()
        for synonym in payment_synonyms:
            if synonym in line_text:
                fields["payment_mode"] = synonym.upper()
                break
        if "payment_mode" in fields:
            break
    
    # --- CURRENCY ---
    # Check for currency indicators in the full text
    full_text = ""
    for region in regions.values():
        for line in region:
            full_text += " ".join([w["text"] for w in line]) + " "
    
    full_lower = full_text.lower()
    if "rm" in full_lower or "ringgit" in full_lower:
        fields["currency"] = "MYR"
    elif "₹" in full_text or "inr" in full_lower or "rs" in full_lower:
        fields["currency"] = "INR"
    elif "$" in full_text or "usd" in full_lower:
        fields["currency"] = "USD"
    elif "sgd" in full_lower:
        fields["currency"] = "SGD"
    
    # --- DOCUMENT TYPE ---
    if "tax invoice" in full_lower:
        fields["document_type"] = "tax_invoice"
    elif "invoice" in full_lower:
        fields["document_type"] = "invoice"
    else:
        fields["document_type"] = "receipt"
    
    return fields


# Test
from src.layout import pair_label_value

extracted = extract_fields(regions, tess_results)

print("Extracted fields:\n")
for field, value in extracted.items():
    print(f"  {field:<20} {value}")

Extracted fields:

  vendor_name          Guardian Health And Beauty Sdn Phd
  total_amount         38.37
  subtotal             38.37
  date                 19/05/18
  payment_mode         CASH
  currency             INR
  document_type        receipt


In [8]:
def detect_currency(full_text):
    """
    Detect currency from receipt text.
    Uses word-boundary matching to avoid false positives.
    """
    # Check word by word to avoid substring false matches
    words = full_text.lower().split()
    
    # Check for explicit currency mentions
    if "rm" in words or "ringgit" in words or "myr" in words:
        return "MYR"
    if "₹" in full_text or "inr" in words:
        return "INR"
    # "rs" and "rs." only count as INR if they appear as standalone words
    for i, w in enumerate(words):
        if w in ("rs", "rs.") and i + 1 < len(words):
            return "INR"
    if "$" in full_text or "usd" in words:
        return "USD"
    if "sgd" in words:
        return "SGD"
    if "s$" in full_text:
        return "SGD"
    
    # Fallback: check for RM in the OCR results even if mangled
    # "RH" is a common Tesseract error for "RM"
    if "rh" in words:
        return "MYR"
    
    return "UNKNOWN"


# Test
full_text = ""
for region in regions.values():
    for line in region:
        full_text += " ".join([w["text"] for w in line]) + " "

currency = detect_currency(full_text)
print(f"Detected currency: {currency}")

Detected currency: MYR


In [9]:
import importlib
import src.extraction
importlib.reload(src.extraction)

from src.extraction import extract_fields as ef

test_extracted = ef(regions)

print("Extracted fields:\n")
for field, value in test_extracted.items():
    print(f"  {field:<20} {value}")

Extracted fields:

  vendor_name          Guardian Health And Beauty Sdn Phd
  total_amount         38.37
  subtotal             38.37
  date                 19/05/18
  payment_mode         CASH
  currency             MYR
  document_type        receipt


In [10]:
import json

def run_pipeline_single(image_path, engine="tesseract"):
    """
    Full pipeline: image → preprocessed → OCR → layout → extraction → JSON
    """
    image = cv2.imread(str(image_path))
    if image is None:
        return {"error": f"Could not load {image_path}"}
    
    image_height = image.shape[0]
    
    # Stage 2: Preprocess
    preprocessed = preprocess(image)
    
    # Stage 3: OCR
    if engine == "tesseract":
        ocr_results = run_tesseract(preprocessed)
    else:
        ocr_results = run_paddleocr(image_path)
    
    # Stage 4: Layout
    from src.layout import group_into_lines, identify_regions
    lines = group_into_lines(ocr_results)
    regions = identify_regions(lines, image_height)
    
    # Stage 5: Extraction
    from src.extraction import extract_fields
    fields = extract_fields(regions)
    
    # Add metadata
    fields["source_file"] = str(Path(image_path).name)
    fields["ocr_engine"] = engine
    
    return fields


# Test on our familiar image
result = run_pipeline_single(image_path, engine="tesseract")

print(json.dumps(result, indent=2))

{
  "vendor_name": "Guardian Health And Beauty Sdn Phd",
  "total_amount": "38.37",
  "subtotal": "38.37",
  "date": "19/05/18",
  "payment_mode": "CASH",
  "currency": "MYR",
  "document_type": "receipt",
  "source_file": "sroie_05.jpg",
  "ocr_engine": "tesseract"
}


In [12]:
import json

results_file = project_root / "outputs" / "all_results_tesseract.json"
with open(results_file) as f:
    all_results = json.load(f)

print(f"Total results: {len(all_results)}\n")

# Show key fields for each result
print(f"{'File':<20} {'Vendor':<35} {'Total':<10} {'Date':<12} {'Currency':<8} {'Type'}")
print("-" * 100)

for r in all_results:
    name = r.get("source_file", "?")[:19]
    vendor = r.get("vendor_name", "—")[:34]
    total = r.get("total_amount", "—")[:9]
    date = r.get("date", "—")[:11]
    currency = r.get("currency", "—")[:7]
    doc_type = r.get("document_type", "—")
    print(f"{name:<20} {vendor:<35} {total:<10} {date:<12} {currency:<8} {doc_type}")

Total results: 42

File                 Vendor                              Total      Date         Currency Type
----------------------------------------------------------------------------------------------------
personal_01.jpg      AUTOMARK MOTORS LIMITED             715.28     —            UNKNOWN  tax_invoice
personal_02.jpg      —                                   —          —            UNKNOWN  receipt
personal_03.jpg      —                                   —          —            UNKNOWN  receipt
personal_04.jpg      —                                   —          —            UNKNOWN  receipt
personal_05.png      SHOPPERS STOP                       —          —            UNKNOWN  receipt
personal_06.jpg      Uka Tarsadia University, Bardoli    —          —            UNKNOWN  receipt
personal_07.jpg      Bardoli                             —          —            UNKNOWN  receipt
personal_08.jpg      —                                   —          —            UNKNOWN  recei

In [14]:
import importlib
import src.extraction
importlib.reload(src.extraction)

from src.extraction import fix_numeric_chars

test_cases = [
    ("3B.37", "38.37"),
    ("4O.35", "40.35"),
    ("I2.00", "12.00"),
    ("38.37", "38.37"),
]

print("Character substitution tests:\n")
for input_val, expected in test_cases:
    result = fix_numeric_chars(input_val)
    status = "✓" if result == expected else "✗"
    print(f"  {status}  '{input_val}' → '{result}'  (expected '{expected}')")

Character substitution tests:

  ✓  '3B.37' → '38.37'  (expected '38.37')
  ✓  '4O.35' → '40.35'  (expected '40.35')
  ✓  'I2.00' → '12.00'  (expected '12.00')
  ✓  '38.37' → '38.37'  (expected '38.37')


In [15]:
importlib.reload(src.extraction)
from src.extraction import extract_fields

# Need to rebuild regions from current data
from src.layout import group_into_lines, identify_regions
tess_lines = group_into_lines(tess_results)
regions = identify_regions(tess_lines, image_height)

result = extract_fields(regions)
print(json.dumps(result, indent=2))

{
  "vendor_name": "Guardian Health And Beauty Sdn Phd",
  "total_amount": "38.37",
  "subtotal": "38.37",
  "date": "19/05/18",
  "payment_mode": "CASH",
  "currency": "MYR",
  "document_type": "receipt",
  "confidence_notes": {
    "vendor_name": "high",
    "total_amount": "high",
    "subtotal": "high",
    "date": "medium"
  }
}


In [16]:
results_file = project_root / "outputs" / "all_results_both.json"
with open(results_file) as f:
    all_results_both = json.load(f)

print(f"{'File':<20} {'Vendor':<35} {'Total':<10} {'Date':<12} {'Currency':<8} {'Type'}")
print("-" * 100)

for r in all_results_both:
    name = r.get("source_file", "?")[:19]
    vendor = r.get("vendor_name", "—")[:34]
    total = r.get("total_amount", "—")[:9]
    date = r.get("date", "—")[:11]
    currency = r.get("currency", "—")[:7]
    doc_type = r.get("document_type", "—")
    print(f"{name:<20} {vendor:<35} {total:<10} {date:<12} {currency:<8} {doc_type}")

# Count improvements
agree_count = 0
disagree_count = 0
for r in all_results_both:
    notes = r.get("confidence_notes", {})
    for key, val in notes.items():
        if key.endswith("_agreement"):
            if val == "agree":
                agree_count += 1
            else:
                disagree_count += 1

print(f"\n--- Agreement stats ---")
print(f"Fields where engines agree:    {agree_count}")
print(f"Fields where engines disagree: {disagree_count}")

File                 Vendor                              Total      Date         Currency Type
----------------------------------------------------------------------------------------------------
personal_01.jpg      AUTOMARK MOTORS LIMITED             715.28     —            UNKNOWN  tax_invoice
personal_02.jpg      —                                   —          —            UNKNOWN  receipt
personal_03.jpg      AUTOMARK MOTORS LIMITED             287.4      —            UNKNOWN  receipt
personal_04.jpg      ASHIRWAD                            —          —            UNKNOWN  receipt
personal_05.png      SHOPPERS STOP                       —          11-11-2025   UNKNOWN  receipt
personal_06.jpg      Uka Tarsadia University, Bardoli    —          —            UNKNOWN  receipt
personal_07.jpg      Bardoli                             —          —            UNKNOWN  receipt
personal_08.jpg      Uka Tarsadia University, Bardoli    —          —            UNKNOWN  receipt
personal_09.jpg 

In [17]:
sroie_dir = project_root / "sroie_txt"
txt_files = sorted(sroie_dir.glob("*.txt"))

print(f"Found {len(txt_files)} txt files\n")
print("First 5 files:")
for f in txt_files[:5]:
    print(f"\n--- {f.name} ---")
    print(f.read_text(encoding="utf-8"))

Found 28 txt files

First 5 files:

--- sroie_01.txt ---
{
    "company": "MR. D.I.Y. (M) SDN BHD",
    "date": "14-03-18",
    "address": "LOT 1851-A & 1851-B, JALAN KPB 6, KAWASAN PERINDUSTRIAN BALAKONG, 43300 SERI KEMBANGAN, SELANGOR",
    "total": "37.10"
}

--- sroie_02.txt ---
{
    "company": "PERNIAGAAN ZHENG HUI",
    "date": "15/03/2018",
    "address": "NO.59 JALAN PERMAS 9/5 BANDAR BARU PERMAS JAYA 81750 JOHOR BAHRU",
    "total": "8.00"
}

--- sroie_03.txt ---
{
    "company": "BEYOND BROTHERS HARDWARE",
    "date": "14/03/2018",
    "address": "LOT 1-0-2, JLN 1/50, DIAMOND SQUARE, OFF JLN GOMBAK 53000 KUALA LUMPUR",
    "total": "599.45"
}

--- sroie_04.txt ---
{
    "company": "YONG SOON FATT S/B",
    "date": "6/2/2017",
    "address": "LOT 1504, BATU 8 1/2, JALAN KLANG LAMA, 46000 PETALING JAYA, SELANGOR.",
    "total": "758.70"
}

--- sroie_05.txt ---
{
    "company": "GUARDIAN HEALTH AND BEAUTY SDN BHD",
    "date": "19/05/18",
    "address": "JALAN LOKE YEW BENTONG 

In [18]:
import json

sroie_dir = project_root / "sroie_txt"
txt_files = sorted(sroie_dir.glob("*.txt"))

ground_truth = {}

for txt_file in txt_files:
    # Read the SROIE label
    data = json.loads(txt_file.read_text(encoding="utf-8"))
    
    # Map SROIE field names to our field names
    image_name = txt_file.stem + ".jpg"  # sroie_01.txt → sroie_01.jpg
    
    ground_truth[image_name] = {
        "vendor_name": data.get("company", ""),
        "date": data.get("date", ""),
        "total_amount": data.get("total", ""),
        "address": data.get("address", ""),
    }

# Save to project root
gt_path = project_root / "ground_truth.json"
with open(gt_path, "w") as f:
    json.dump(ground_truth, f, indent=2)

print(f"Ground truth saved: {gt_path}")
print(f"Images covered: {len(ground_truth)}")
print(f"\nSample (sroie_05.jpg):")
print(json.dumps(ground_truth["sroie_05.jpg"], indent=2))

Ground truth saved: d:\Projects\ocr-document-understanding\ground_truth.json
Images covered: 28

Sample (sroie_05.jpg):
{
  "vendor_name": "GUARDIAN HEALTH AND BEAUTY SDN BHD",
  "date": "19/05/18",
  "total_amount": "38.37",
  "address": "JALAN LOKE YEW BENTONG 90 GROUND FLOOR JALAN BENTONG"
}
